<a href="https://colab.research.google.com/github/casthielcastro/Desafio_01_Meteorologia/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gspread faker

In [2]:
import json, gspread
from google.colab import userdata
from faker import Faker
import random

info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@projeto-meteorologia.iam.gserviceaccount.com


In [3]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "18k9DNEyGZx9jOPZrFWdlp8RO3asxBdvR3fJDLduNgKo"
NOME_TABELA = "estacoes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Meteorologia | estacoes | https://docs.google.com/spreadsheets/d/18k9DNEyGZx9jOPZrFWdlp8RO3asxBdvR3fJDLduNgKo


In [4]:
COLUNAS = ["estacao_id", "nome_estacao", "cidade", "pais",
           "latitude", "longitude", "altitude_m"]

ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [5]:
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    return {
        "estacao_id": i,
        "nome_estacao": f"EST-{i:05d}",
        "cidade": fake.city(),
        "pais": "Brasil",
        "latitude": round(random.uniform(-33, 5), 4),
        "longitude": round(random.uniform(-73, -35), 4),
        "altitude_m": random.randint(0, 3499),
    }

linhas = [gerar_linha(i) for i in range(1, 11)]
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [6]:
valores_lidos = ws.get_all_values()
print("Linhas (cabeçalho + dados):", len(valores_lidos))
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['estacao_id', 'nome_estacao', 'cidade', 'pais', 'latitude', 'longitude', 'altitude_m'], ['1', 'EST-00001', 'Alves', 'Brasil', '-8,7018', '-72,0496', '1126'], ['2', 'EST-00002', 'da Cruz de Goiás', 'Brasil', '-23,6941', '-67,6976', '419']]


In [7]:
import time
import gspread
import pandas as pd

def df_para_valores(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)
    lidas = len(ws.col_values(1)) - 1
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")
    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

In [8]:
from google.colab import files
files.upload()

BASES = {
    "estacoes": pd.read_csv("estacoes.csv"),
    "leituras": pd.read_csv("leituras.csv"),
}
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving estacoes.csv to estacoes.csv
Saving leituras.csv to leituras.csv
{'estacoes': 10050, 'leituras': 62000} | total: 72050


In [9]:
for nome, df in BASES.items():
    enviar(sh, df, nome)
remover_aba_padrao(sh)

  estacoes: 5.000 / 10.050
  estacoes: 10.000 / 10.050
  estacoes: 10.050 / 10.050
OK estacoes: 10.050 linhas na planilha
  leituras: 5.000 / 62.000
  leituras: 10.000 / 62.000
  leituras: 15.000 / 62.000
  leituras: 20.000 / 62.000
  leituras: 25.000 / 62.000
  leituras: 30.000 / 62.000
  leituras: 35.000 / 62.000
  leituras: 40.000 / 62.000
  leituras: 45.000 / 62.000
  leituras: 50.000 / 62.000
  leituras: 55.000 / 62.000
  leituras: 60.000 / 62.000
  leituras: 62.000 / 62.000
OK leituras: 62.000 linhas na planilha


In [10]:
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"
print("A planilha confere com a base gerada.")

Total de linhas: 72.050
A planilha confere com a base gerada.
